In [2]:
import io
import requests
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import time
import pickle as pkl
import torch
from cffi.backend_ctypes import long
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "src/database/BindingDB_All_202605_tsv.zip"
NOTEBOOK_DATAFOLDER = PROJECT_ROOT / "notebook_database"


In [2]:
# Goal: get protein sequence + smiles pairs
# task: get pocket patch proteins - need to isolate pocket sequence via UniProt + PDB
# BindingDB: Protein-Ligand affinity data + UniProt IDs
# PDB: will contain information on proteins close to drug (6-8 A)

In [13]:
# Data settings
target_fields = [
    "Ligand SMILES",
    "PDB ID(s) of Target Chain 1",
    "Ki (nM)",  # inhibitor binding strength -> both Ki and Kd are good since we want POCKETS
]

nm_threshold = 1   # isolate high affinity pairs
chunksize = 100000
pdb_resolution = 2.5

In [5]:
chunks = []

with zipfile.ZipFile(DATAPATH) as z_fil:
    tsv_data = [f for f in z_fil.namelist() if f.endswith(".tsv")][0]
    with z_fil.open(tsv_data) as f:
        reader = pd.read_csv(
            f,
            sep="\t",
            dtype=str,
            usecols=target_fields,
            low_memory=False,
            on_bad_lines="skip",
            chunksize=chunksize,
        )
        for chunk in reader:
            filtered = chunk[target_fields]
            chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)

In [15]:
# I only want potent binders
print(df.shape)  # 3176528, 3


# apply potency filter -> can adjust nm_threshold down the line -> okay definitely adjusting
# I'm getting way too much data on me, just simply not feasible
df["Ki (nM)"] = pd.to_numeric(df["Ki (nM)"], errors="coerce")
potent_df = df[df["Ki (nM)"] <= nm_threshold]
print(potent_df.shape)   # 59283, 3

potent_df.head(3)

(3176528, 3)
(59283, 3)


,Ligand SMILES,PDB ID(s) of Target Chain 1,Ki (nM)
0,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCC(O)=...,"1W5Y,1W5X,1W5W,1W5V,2FDE,7UPJ,6UWC,6UWB,6D0E,6...",0.24
1,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(C\C=C\c2cn...,"1W5Y,1W5X,1W5W,1W5V,2FDE,7UPJ,6UWC,6UWB,6D0E,6...",0.25
2,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CC2CC2)C(=...,"1W5Y,1W5X,1W5W,1W5V,2FDE,7UPJ,6UWC,6UWB,6D0E,6...",0.41


In [16]:
unique_smiles_strings = potent_df["Ligand SMILES"].dropna().unique()

all_smiles = set()
for smile in unique_smiles_strings:
    all_smiles.add(smile)

print(len(all_smiles))  # 36437
unique_smiles_strings = pd.DataFrame(all_smiles)
unique_smiles_strings.to_csv("notebook_database/unique_smiles_strings.csv")

36437


In [17]:
unique_pdb_strings = potent_df["PDB ID(s) of Target Chain 1"].dropna().unique()
# print(len(unique_pdb_strings))  # --> 1985 string clusters

all_pdb_ids = set()

for pdb_string in unique_pdb_strings:
    for pid in pdb_string.split(","):
        all_pdb_ids.add(pid.strip())

print(len(all_pdb_ids)) # --> 30259

19632


In [18]:
# this took me 2 hours to run but im pretty sure with the multithreading thing it can be accelerated to like 5 mins. idk how right now though maybe someone can help figure that out

def get_pocket(pdb_id):
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.upper()}"
    try:
        req = requests.get(url, timeout=10)
        if req.status_code != 200:
            return None
        data = req.json()

        req_data = data.get("rcsb_entry_info", {})

        return {
            "pdb_id": pdb_id.upper(),
            "resolution": req_data.get("resolution_combined", [999])[0],
            "ligand_count": req_data.get("nonpolymer_entity_count", 0),
        }

    except Exception as e:
        print(f"Couldn't get info on {pdb_id} : {e}")
        return None


def pdb_filters(all_pdb_ids, delay=0.1):
    result_df = []
    for pdb_id in tqdm(all_pdb_ids, total=len(all_pdb_ids), desc="Validating PDBs + Building Cache"):
        meta_data = get_pocket(pdb_id)

        # apply filters -> want relatively high resolution and a sequence that's acc bound to smth
        if meta_data is None:
            continue

        time.sleep(delay)

        if meta_data["resolution"] > 2.5:
            continue
        if meta_data["ligand_count"] == 0:
            continue

        result_df.append({
            "PDB ID": pdb_id,
        })

    return pd.DataFrame(result_df)

pdb_df = pdb_filters(all_pdb_ids=all_pdb_ids, delay=0.1)
pdb_df.to_csv("notebook_database/unique_pdb_cache.csv")

Validating PDBs + Building Cache: 100%|██████████| 19632/19632 [1:27:44<00:00,  3.73it/s]


In [19]:
# Ok so pdb_df is gonna contain all valid unique pdb IDs
# potent_df is gonna contain all SMILES
# going to need to switch to mmCIF but that's fine - same 4 char codes
# isolate atoms within 8 angstroms of ligand - look for LIG comp_id
# we're getting relative position vectors to the centroid of the ligand
# this means in data generation, we would first have to compute the SMILES's 3d structure
#
# training: we're passing relative coords. to a centroid that's already calc., model would learn this implicitly during training via cross attention
# probably wouldn't hurt to pass centroid vector through regardless - we'll see
# save it just in case ^^
# so far the only reason this severely matters is data visualization, gonna need to relatively position the AA's around a calculated centroid
# would probably have an entirely different data visualizer module

# lemme just place this here to get the functions defined
# will call on it using thread in another cell

# notes from biopython cookbook
# structure consists of models
# model consists of chains
# chain consists of residues
# residue consists of atoms

# atom: atom obj: atom id simply is atom name
# Ca (alpha) atoms are called '.CA.'
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB import Selection
from Bio.PDB.Polypeptide import PPBuilder

# load all unique pdb ids as a list
pdb_df = pd.read_csv("notebook_database/unique_pdb_cache.csv")
pdb_list = pdb_df["PDB ID"].to_list()

# Residues to exclude: waters, common ions, solvents
EXCLUDE_HETS = {
    "HOH", "WAT", "DOD",  # water
    "NA", "K", "MG", "CA", "ZN", "FE", "MN", "CU", "CL", "BR", "I",  # ions
    "SO4", "PO4", "GOL", "EDO", "PEG", "MPD", "DMS", "ACT", "ACE",   # solvents/buffers
    "NO3", "CO3", "EPE", "MES", "TRS", "HEZ"
}

AA_codetoletter = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
    "MSE": "selM",  # selenomethionine: gonna need to do some literature on these later. ok thank god they're not included in anything
    "SEP": "phoS",  # phosphoserine
    "TPO": "phoT",  # phosphothreonine
    "PTR": "phoY",  # phosphotyrosine
    "HYP": "hydP",  # hydroxyproline
}

ANGSTROM_THRESHOLD = 8.0


def fetch_struct(pdb_id: str):
    """I want to avoid disk writing, it's like 20k entries"""
    url = f"https://files.rcsb.org/download/{pdb_id.lower()}.cif"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return None

    cif_handle = io.StringIO(r.text)
    parser = MMCIFParser(QUIET=True)
    struct = parser.get_structure(pdb_id, cif_handle)
    return struct

def get_ligand_atoms(struct):
    """Extract HETATM belonging to drug-like ligands
    returns: best_coords, resname
    """
    best_coords = None
    best_count = 0

    model = next(iter(struct))

    for chain in model:
        for residue in chain:
            hetflag, _, _ = residue.id
            if not hetflag.startswith("H_"):
                continue
            resname = residue.resname.strip()
            if resname in EXCLUDE_HETS:
                continue

            heavy = np.array([
                atom.get_vector().get_array() for atom in residue.get_atoms() if atom.element != "H"
            ])

            if len(heavy) > best_count:
                best_count = len(heavy)
                best_coords = heavy


    if best_coords is None:
        return None

    return best_coords


# def get_ligand_data(ligand_atoms):
#     """get the geometric center (avg. pos) of all atoms within the ligand mol"""
#     coords = np.array([atom.get_vector().get_array()
#                        for atom, _ in ligand_atoms])
#     return coords, coords.mean(axis=0)

def get_pocket_residues(struct, ligand_coords, threshold: float = ANGSTROM_THRESHOLD):
    """
    for each protein residue, get their min distance to any ligand atom
    return residues within angstrom_threshold, get alpha carbon vector relative to ligand centroid
    :param struct:
    :param ligand_coords:
    :param threshold:
    :return:
    """
    ligand_centroid = ligand_coords.mean(axis=0)
    AA_ids, threeD_coords = [],[]

    model = next(iter(struct))

    # residue.id is a tuple of (hetflag, sequence_number, insertion_code)
    for chain in model:
        for residue in chain:
            if residue.id[0] != " ":  # blank for standard amino and nucleic acids
                continue
            if "CA" not in residue:  # this is where we get the coordinate vector from
                continue
            if residue.resname.strip() not in AA_codetoletter:
                continue

            res_heavy = np.array([
                atom.get_vector().get_array() for atom in residue.get_atoms() if atom.element != "H"
            ])
            # collect 3D coordinates of every non-hydrogen atom in the residue
            # cleaner, more efficient + doesn't contribute unique topological information


            # don't focus just on alpha Carbon at this stage since we want all proteins within that threshold distance FIRST

            # res_heavy is shape (N_res_atoms, 3)
            # ligand_coords is shape (N_lig_atoms, 3)
            # np.newaxis reshapes them so NumPy broadcasts subtraction across every pair
            # basically multiplying the matrices so they line up for subtraction
            diffs = res_heavy[:, np.newaxis, :] - ligand_coords[np.newaxis, :, :]
            min_dist = np.sqrt((diffs**2).sum(axis=2)).min()

            if min_dist <= threshold:
                ca_coord = residue["CA"].get_vector().get_array()
                relative_pos = ca_coord - ligand_centroid  # obtain AA vector relevant to ligand

                AA_ids.append(AA_codetoletter[str(residue.resname)])
                threeD_coords.append(relative_pos)

    # property dict
    pocket_patch_residues = {
        "AA_IDs": AA_ids,   # indices align
        "3d_struct": threeD_coords
    }

    return pocket_patch_residues


def pdb_miner(pdb_id: str):
    try:
        struct = fetch_struct(pdb_id)
        if struct is None:
            return None

        ligand_coords = get_ligand_atoms(struct)
        if ligand_coords is None:
            return None

        residues = get_pocket_residues(struct, ligand_coords)


        if not residues["AA_IDs"]:
            return None

        result = {
            "PDB ID": pdb_id,
            "AA_IDs": residues["AA_IDs"],
            "3d_struct": residues["3d_struct"]
        }

        # get this in one clean dataframe instead of having to work around it

        return result

    except Exception as e:   # way too many error messages, just check the yield
        # we've done enough filtering already.
        return None

In [20]:
# ok use this to run the above functions
import warnings
from Bio import BiopythonWarning

from concurrent.futures import ThreadPoolExecutor, as_completed

# they're going to warn us saying we have discontinuous chains
# this is fine, we're not relying on contiguous sequence, we're relying on 3D spatially encoded sequence, whose quality we're already filtering
warnings.filterwarnings("ignore", category=BiopythonWarning)

def pocket_pipeline(pdb_ids: list, max_workers):
    AA3D_df = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(pdb_miner, pdb_id): pdb_id for pdb_id in pdb_ids}

        for future in tqdm(as_completed(futures), total=len(pdb_ids)):
            result = future.result()
            if result:
                AA3D_df.append(result)

    return pd.DataFrame(AA3D_df)

# bottleneck is network I/O, so batch processing won't do much. save that for training / tokenizing
# takes about an hour depending on what you set max_workers to.
AA3D_df = pocket_pipeline(pdb_ids=pdb_list, max_workers=40)

100%|██████████| 13351/13351 [36:09<00:00,  6.15it/s] 


In [21]:
success = len(AA3D_df)
failed = len(pdb_list) - len(AA3D_df)
fail_rate = failed / len(pdb_list)
print(f"datamining failrate = {fail_rate}")  # 0.04704

AA3D_df.to_csv("notebook_database/AA3D_df.csv")
# update: looking much cleaner

datamining failrate = 0.047037675080518315


In [22]:
# MILESTONE 1: cleaned/processed + isolated protein binding pocket coordinates for all unique PBD IDs

In [23]:
AA3D_df = pd.read_csv("notebook_database/AA3D_df.csv")
# keys: PDB ID and pocket_AA_data
#                  |--> within pocket_AA_data there is:
#                   AA_IDs and 3d_struct -> amino acid code and 3d vectors

In [56]:
from rdkit import Chem
from map4 import MAP4
# look at study one molecular fingerprint to rule them all: drugs, biomolecules
# can always do a double take later

mol_dim = 1024
map4_fprinter = MAP4(
    dimensions=mol_dim,
    radius=2,
    include_duplicated_shingles=True,
)

def mol_from_smiles(smiles):
    """Extract molecular fingerprint from singular SMILES"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        map4_fp = map4_fprinter.calculate(mol=mol)

        return map4_fp

    except Exception as e:
        return None


## TEST BLOCK
# test_smiles = "COc1nc(OC[C@H]2C[C@@H]2c2ccc3ncccc3n2)nc(NCc2cnn(C)c2)c1C |r|"
# test_fp = mol_from_smiles(test_smiles)

# test_fp  #array([1, 0, 1, ..., 0, 0, 0], shape=(1024,), dtype=uint8)

In [51]:
unique_smiles_strings_df = pd.read_csv("notebook_database/unique_smiles_strings.csv")

MOLFP_df = []
idxs = 0
fails = 0
for _, row in tqdm(unique_smiles_strings_df.iterrows(), total=len(unique_smiles_strings), desc="Generating Fingerprints"):
    smiles = str(row["0"])
    fingerprint = mol_from_smiles(smiles=smiles)
    if fingerprint is None:
        fails += 1
        continue

    MOLFP_df.append({
        "smiles_str": str(smiles),
        "map4_fp": fingerprint,
    })
    idxs += 1

# should put in sanitization
failrate = float(fails / idxs)  # check for systemic errors
print(failrate)  # 0.029% rounded up so we're good

Generating Fingerprints:   5%|▌         | 1919/36437 [00:08<02:26, 235.64it/s][07:32:02] 

****
Pre-condition Violation
bad bond stereo
Violation occurred on line 362 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/Canon.cpp
Failed Expression: dblBond->getStereo() > Bond::STEREOANY
----------
Stacktrace:
----------
****

Generating Fingerprints:   8%|▊         | 2981/36437 [00:14<02:47, 199.48it/s][07:32:07] 

****
Pre-condition Violation
bad bond stereo
Violation occurred on line 362 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/Canon.cpp
Failed Expression: dblBond->getStereo() > Bond::STEREOANY
----------
Stacktrace:
----------
****

Generating Fingerprints:  16%|█▌        | 5666/36437 [00:27<02:30, 204.06it/s][07:32:21] 

****
Pre-condition Violation
bad bond stereo
Violation occurred on line 362 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/Canon.cpp
Failed Expression: dblBond->getStereo() > Bond::STEREO

0.030225062203121465


In [58]:
print(len(MOLFP_df))

MOLFP_df = pd.DataFrame(MOLFP_df)
MOLFP_df["map4_fp"] = MOLFP_df["map4_fp"].apply(lambda x: torch.from_numpy(x).float())
MOLFP_df.to_pickle("notebook_database/molfp_df.pkl")

35368


In [65]:
pickled_fp = pd.read_pickle(Path("notebook_database/molfp_df.pkl"))
sample = pickled_fp["map4_fp"].iloc[0]
print(type(sample))

<class 'torch.Tensor'>


In [ ]:
# MILESTONE 2: obtained fingerprints for all unique SMILES

In [67]:
# one more thing to do: create training information csv
# col 1: all unique SMILES
# col 2: all proteins that were tested on it
# mental note: remember they're all here because they had high binding affinity

initial_info = df.dropna(subset=["PDB ID(s) of Target Chain 1"])
unique_smiles = pd.read_csv("notebook_database/unique_smiles_strings.csv")
unique_pdb_ids = pd.read_csv("notebook_database/unique_pdb_cache.csv")

# initial_info.columns
# ['Ligand SMILES', 'PDB ID(s) of Target Chain 1', 'Ki (nM)']
# PDB column contains a string cluster of all PDB IDs for that - separated ','

# unique_smiles: smile strings are in unique_smiles["0"]...

# unique_pdb_ids: PDB IDs are in unique_pdb_ids["PDB ID"]


def get_all_pairs():
    """
    Iterate through the unique pdb ids first
    Obtain each ID's SMILES hit through 'initial_info'
    Doing this because we want to train test split by protein not molecule
    Wanna see if it'll generate proteins that max. biochemical / structural properties required to bind this molecule? or if it overtrained to see one specific pattern
    :return:
    """
    unique_pdb_set = set(unique_pdb_ids["PDB ID"].astype(str).str.strip())
    unique_smiles_set = set(unique_smiles["0"].astype(str).str.strip())

    df = initial_info.dropna(subset=["PDB ID(s) of Target Chain 1"]).copy()
    # Filter to only SMILES we actually want / have now
    df = df[df["Ligand SMILES"].isin(unique_smiles_set)]

    # Explode multi-PDB column
    df["pdb_list"] = df["PDB ID(s) of Target Chain 1"].str.split(",")
    exploded = df.explode("pdb_list").copy()

    exploded["pdb_list"] = exploded["pdb_list"].str.strip()

    # Keep only relevant PDBs and relevant SMILES (already filtered)
    exploded = exploded[exploded["pdb_list"].isin(unique_pdb_set)]
    exploded = exploded.drop_duplicates(subset=["pdb_list", "Ligand SMILES"])

    # Group SMILES per PDB
    pdb_to_smiles = exploded.groupby("Ligand SMILES")["pdb_list"].apply(list)

    # Create final dataframe with ALL unique PDBs
    result = pd.DataFrame({
        "SMILES": list(pdb_to_smiles.index.tolist())
    })

    result["PDB_hits"] = result["SMILES"].map(pdb_to_smiles)
    result["PDB_hits"] = result["PDB_hits"].apply(lambda x: x if isinstance(x, list) else [])

    return result


all_pairs_df = get_all_pairs()

# Quick summary
print(all_pairs_df.head(10))
print(f"\nTotal smiles: {len(all_pairs_df)}")
print(f"\nUnique smiles: {len(unique_smiles)}")
# dataframes line up ok we're good. great use of 2 hours of my time.

                                              SMILES  \
0     B(CNS(=O)(=O)c1ccc(cc1C(F)(F)F)c2[nH]nnn2)(O)O   
1          B1([C@@H]2C[C@@H]2c3ccc(c(c3O1)C(=O)O)F)O   
2         BrCc1ccc(nc1)-c1cnc(o1)C(=O)CCCCCCc1ccccc1   
3            BrN1C[C@H]2C[C@H](C1)c1cccc(=O)n1C2 |r|   
4              Brc1c(I)cc2[C@@H]3CNCC(C3)Cn2c1=O |r|   
5  Brc1cc(C[C@@H](NC(=O)N2CCC(CC2)N2Cc3ccccc3NC2=...   
6  Brc1cc(NS(=O)(=O)c2ccccc2)ccc1OC[C@H]1CC[C@H](...   
7  Brc1cc(cs1)C(=O)N[C@@H]1C2CCN(CC2)[C@@H]1Cc1cc...   
8    Brc1cc2CCN(CCN3CCN(CC3)c3nsc4ccccc34)C(=O)c2cn1   
9  Brc1cc2CC[C@@H](CC(=O)Nc3ccccc3)n3c2c(c1)[nH]c...   

                                            PDB_hits  
0  [9DHL, 9C84, 9C81, 9C6P, 6YPD, 6YEO, 6YEN, 6TP...  
1  [9ZOW, 9ZOV, 9ZOU, 9ZOT, 9ZOS, 9ZOR, 9ZOQ, 9ZO...  
2                                       [3QK5, 3QJ9]  
3   [8V8A, 8V88, 8V86, 8V80, 8UZJ, 8UTB, 8UT1, 8C9X]  
4  [8ST0, 8ST4, 8V8A, 8V88, 8V86, 8V80, 8UZJ, 8UT...  
5                                       [3NXU, 3N7S] 

In [68]:
all_pairs_df = pd.DataFrame(all_pairs_df)
all_pairs_df.to_csv("notebook_database/SMILE_2_PDBhits.csv")